# Programação Orientada a Objetos com Python — Parte 4
## Tópicos Avançados e Boas Práticas

**Tema: O Podrão — Sistema de Lanchonete / Food Truck**

**Disciplina:** COMP0496 — Programação A
**Duração:** 200 minutos (Aula 4 de 4 do módulo de POO)

---

### Pré-requisitos

Você deve ter concluído as Partes 1, 2 e 3 e estar confortável com:

- Classes, `__init__`, `self`, atributos de instância e de classe.
- `@property`, setters, propriedades read-only, métodos especiais (`__str__`, `__repr__`, `__eq__`, `__lt__`).
- Herança, sobrescrita, polimorfismo, composição, classes abstratas.

### Objetivos desta aula

Ao final desta aula, você será capaz de:

1. Usar `@dataclass` para eliminar *boilerplate* e criar classes de dados idiomáticas.
2. Distinguir **métodos de instância**, `@classmethod` e `@staticmethod` e saber quando usar cada um.
3. Reconhecer violações dos princípios **SRP**, **OCP** e **LSP** e saber como corrigi-las.
4. Implementar os padrões **Strategy** e **Factory Method** em suas versões Pythônicas.
5. **Refatorar** um sistema procedural mal projetado aplicando os conceitos das quatro aulas.

### Como usar este material

Esta é a aula mais densa do módulo — não por introduzir muitos conceitos novos, mas por exigir que você **integre** tudo o que aprendeu. Siga na ordem e não pule o estudo de caso final: é ele que consolida tudo.

---

## 1. Revisão rápida das Aulas 1–3

Antes de começar, vamos lembrar onde estamos. Se você construísse hoje um `Lanche` completo com tudo o que vimos, ele seria mais ou menos assim:

In [ ]:
class Lanche:
    """Um Lanche "completo" com tudo que aprendemos até agora."""

    taxa_servico = 0.10                       # (Aula 1) atributo de classe

    def __init__(self, nome, preco, ingredientes):
        self.nome = nome                      # (Aula 1) atributo de instância
        self._preco = preco                   # (Aula 2) protegido + @property
        self.ingredientes = ingredientes

    @property                                 # (Aula 2) propriedade com validação
    def preco(self):
        return self._preco

    @preco.setter
    def preco(self, novo):
        if novo < 0:
            raise ValueError("Preço não pode ser negativo.")
        self._preco = novo

    @property                                 # (Aula 2) read-only derivada
    def preco_final(self):
        return self._preco * (1 + Lanche.taxa_servico)

    def __str__(self):                        # (Aula 2) dunder
        return f"🍔 {self.nome} — R${self.preco_final:.2f}"

    def __eq__(self, outro):                  # (Aula 2) igualdade por valor
        if not isinstance(outro, Lanche):
            return NotImplemented
        return self.nome == outro.nome and self._preco == outro._preco

    def __hash__(self):
        return hash((self.nome, self._preco))


x = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
print(x)
print(f"Preço base: R${x.preco:.2f}")
print(f"Preço final (com taxa): R${x.preco_final:.2f}")


Isso funciona. Mas observe: para criar uma classe relativamente simples, foram ~30 linhas. E a maior parte delas é **código repetitivo** — o mesmo padrão de `__init__` guardando parâmetros em `self`, o mesmo `__eq__` comparando atributos, o mesmo `__hash__`…

Vamos começar a aula aprendendo a eliminar essa repetição.

---

## 2. `@dataclass`: eliminando o *boilerplate*

A partir do Python 3.7, existe o decorador `@dataclass` (do módulo `dataclasses`) que **gera automaticamente** `__init__`, `__repr__`, `__eq__` (e opcionalmente outros dunders) a partir das *anotações de tipo* dos atributos.

### 2.1 A forma mais simples

In [ ]:
from dataclasses import dataclass


@dataclass
class Lanche:
    nome: str
    preco: float
    ingredientes: str


# Você ganha __init__, __repr__ e __eq__ DE GRAÇA
x = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
d = Lanche("Dogão", 12.00, "pão, salsicha, vinagrete")

print(x)                                      # __repr__ automático
print(f"Preço: R${x.preco:.2f}")              # atributos normais

# __eq__ comparando por valor, automático
y = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo, bacon")
print(f"x == y?  {x == y}")                   # True
print(f"x == d?  {x == d}")                   # False


Compare com a classe "manual" da seção 1: a versão com `@dataclass` é **~5 vezes menor** e faz essencialmente o mesmo. Os dois `print` abaixo ficariam idênticos.

### 2.2 Parâmetros úteis do `@dataclass`

O decorador aceita vários parâmetros que controlam o que é gerado:

| Parâmetro | Padrão | Efeito |
|---|---|---|
| `frozen` | `False` | Se `True`, os atributos viram imutáveis (`dataclass` congelada) |
| `order` | `False` | Se `True`, gera `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `eq` | `True` | Se `False`, não gera `__eq__` |
| `repr` | `True` | Se `False`, não gera `__repr__` |

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True, order=True)
class Lanche:
    """Lanche imutável e ordenável."""
    nome: str
    preco: float
    ingredientes: str = ""                    # valor padrão


# frozen=True → imutável
x = Lanche("X-Tudo", 18.50)
try:
    x.preco = 20.00                           # ❌ vai falhar
except Exception as e:
    print(f"✘ {type(e).__name__}: {e}")

# order=True → dá comparações e sorted() de graça
# (ordena pela primeira coluna, depois pela segunda, etc.)
cardapio = [
    Lanche("Dogão", 12.00),
    Lanche("X-Tudo", 18.50),
    Lanche("Açaí", 15.00),
]

print("\nOrdenado (por nome, depois preço):")
for item in sorted(cardapio):
    print(f"  {item}")


### 2.3 Atributos mutáveis como valor padrão: use `field(default_factory=...)`

Lembra da **armadilha da lista mutável como atributo de classe** da Aula 1? Em `@dataclass` essa armadilha é **detectada automaticamente** e gera um erro. A solução é usar `field(default_factory=list)`.

In [ ]:
from dataclasses import dataclass, field


# ❌ Tentativa ingênua — gera ERRO ao definir a classe
try:
    @dataclass
    class PedidoErrado:
        cliente: str
        itens: list = []                      # ← valor mutável padrão: proibido!
except ValueError as e:
    print(f"✘ {type(e).__name__}: {e}")
    print("  A própria @dataclass recusa esse padrão.")

# ✔ Forma correta
@dataclass
class Pedido:
    cliente: str
    itens: list = field(default_factory=list) # fábrica: cria lista NOVA por instância

    def adicionar(self, lanche):
        self.itens.append(lanche)


p1 = Pedido("João")
p2 = Pedido("Maria")
p1.adicionar(Lanche("X-Tudo", 18.50))

print(f"\np1: {p1}")
print(f"p2: {p2}")
print("✔ Cada pedido tem a sua própria lista independente.")


### 2.4 Adicionando métodos normais

`@dataclass` só gera os *dunders* — você continua livre para adicionar quaisquer métodos que quiser. Dataclasses não são "classes diferentes", são **classes normais** com dunders auto-gerados.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Pedido:
    cliente: str
    itens: list = field(default_factory=list)

    # Métodos normais — você define como sempre
    def adicionar(self, lanche):
        self.itens.append(lanche)

    def total(self):
        return sum(item.preco for item in self.itens)

    def __len__(self):
        return len(self.itens)


p = Pedido("Ana")
p.adicionar(Lanche("X-Tudo", 18.50))
p.adicionar(Lanche("Açaí", 15.00))
print(p)
print(f"Itens no pedido: {len(p)}")
print(f"Total: R${p.total():.2f}")


**Quando usar `@dataclass`?**

- Sempre que sua classe for essencialmente um **contêiner de dados** com poucas regras de negócio.
- Quando você quer `__eq__` e `__repr__` sem ter que pensar.
- Quando quer imutabilidade declarativa (`frozen=True`).

**Quando NÃO usar?**

- Quando a classe tem lógica complexa no `__init__` (validações pesadas, cálculos derivados).
- Quando você precisa de muitas propriedades customizadas (`@property`).
- Quando a classe é fortemente orientada a comportamento, não a dados.

### Exercício 2.1
Transforme sua classe `Bebida` (da Aula 3) em uma `@dataclass`. Ela deve ter `nome: str`, `preco: float`, `volume_ml: int`. Torne-a ordenável por preço usando `order=True`.

Crie uma lista de pelo menos 4 bebidas, ordene-as e imprima o resultado.

In [ ]:
# Escreva sua solução aqui



---
## 3. `@classmethod` e `@staticmethod`

Até agora, todos os métodos que você viu eram **métodos de instância**: recebem `self` e operam sobre um objeto específico. Mas existem dois outros tipos de método ligados à classe:

| Tipo | Primeiro parâmetro | Chamado sobre | Uso típico |
|---|---|---|---|
| **Instância** | `self` | um objeto | a grande maioria dos métodos |
| **`@classmethod`** | `cls` (a própria classe) | classe ou objeto | construtores alternativos, métodos que usam atributos de classe |
| **`@staticmethod`** | (nenhum) | classe ou objeto | funções utilitárias que *logicamente* pertencem à classe mas não usam nem `self` nem `cls` |

### 3.1 `@classmethod` como construtor alternativo

O uso mais comum é criar objetos a partir de **formatos diferentes** do construtor padrão. Exemplo: você tem um arquivo CSV com lanches e quer um método `Lanche.from_csv_line("X-Tudo;18.50;pão,queijo")`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Lanche:
    nome: str
    preco: float
    ingredientes: str

    @classmethod
    def from_csv_line(cls, linha):
        """Cria um Lanche a partir de uma linha CSV: 'nome;preco;ingredientes'."""
        partes = linha.strip().split(";")
        return cls(
            nome=partes[0],
            preco=float(partes[1]),
            ingredientes=partes[2]
        )

    @classmethod
    def from_dict(cls, dados):
        """Cria um Lanche a partir de um dicionário."""
        return cls(**dados)                   # ** descompacta o dict como kwargs


# Uso normal: construtor padrão
x = Lanche("X-Tudo", 18.50, "pão, hambúrguer, queijo")

# Construtores alternativos:
d = Lanche.from_csv_line("Dogão;12.00;pão, salsicha, vinagrete")
a = Lanche.from_dict({"nome": "Açaí 500ml", "preco": 15.00, "ingredientes": "açaí, granola"})

for item in [x, d, a]:
    print(item)


**Por que não usar uma função normal?**

Você poderia escrever `criar_lanche_de_csv(linha)` como função solta fora da classe. Funcionaria. Mas colocar como `@classmethod` dá três vantagens:

1. **Agrupamento lógico**: o código que cria `Lanche` mora junto do resto de `Lanche`. Quem lê o código já sabe onde procurar.
2. **Herança**: se alguém criar `XTudo(Lanche)`, `XTudo.from_csv_line(...)` vai automaticamente criar um `XTudo`, não um `Lanche`. É por isso que o primeiro parâmetro é `cls` (a classe real no momento da chamada), não `Lanche` fixo.
3. **Namespace limpo**: evita poluir o módulo com funções soltas.

Vamos confirmar o ponto 2:

In [ ]:
class XTudo(Lanche):
    pass


# Chamado em XTudo — cls é XTudo, não Lanche
x = XTudo.from_csv_line("X-Tudo;18.50;pão, hambúrguer, queijo, bacon")
print(f"type(x): {type(x).__name__}")           # XTudo, não Lanche
print(f"x é XTudo? {isinstance(x, XTudo)}")


### 3.2 `@staticmethod` para utilitários

O `@staticmethod` é usado quando a função *logicamente* pertence à classe, mas **não precisa** de `self` nem de `cls`. É quase como uma função normal que você decidiu guardar dentro da classe por organização.

In [ ]:
from dataclasses import dataclass


@dataclass
class Lanche:
    nome: str
    preco: float

    @staticmethod
    def calcular_taxa_servico(subtotal, percentual=10):
        """Utilitária: calcula taxa de serviço. Não depende de instância nem classe."""
        return subtotal * (percentual / 100)

    @staticmethod
    def validar_preco(valor):
        """Utilitária: valida se um número é um preço aceitável."""
        return isinstance(valor, (int, float)) and 0 < valor < 1000


# Podem ser chamados pela classe diretamente, sem criar instância
print(f"Taxa sobre R$50: R${Lanche.calcular_taxa_servico(50):.2f}")
print(f"Taxa 15% sobre R$50: R${Lanche.calcular_taxa_servico(50, 15):.2f}")
print(f"10 é preço válido? {Lanche.validar_preco(10)}")
print(f"-5 é preço válido? {Lanche.validar_preco(-5)}")


### 3.3 Escolhendo entre os três

A pergunta-chave é: *"o que o método precisa saber para funcionar?"*

- Se precisa dos **dados do objeto específico** (`self.preco`, `self.nome`) → **método de instância**.
- Se precisa da **classe** mas não de nenhum objeto específico (construtor alternativo, contador, registro) → **`@classmethod`**.
- Se **não precisa** nem de um nem do outro — é pura função utilitária → **`@staticmethod`**.

Na dúvida, use método de instância. `@staticmethod` em particular é frequentemente um sinal de que a função deveria ser só uma função normal no módulo.

### Exercício 3.1
Adicione à sua `Bebida` dataclass:

1. Um `@classmethod` chamado `garrafa_padrao(cls, nome, preco)` que cria uma bebida já com `volume_ml=350` (a "garrafinha" padrão do food truck).
2. Um `@classmethod` chamado `litrao(cls, nome, preco)` que cria uma bebida com `volume_ml=1000`.
3. Um `@staticmethod` chamado `ml_para_litros(ml)` que converte mililitros para litros (`ml / 1000`).

Teste criando bebidas com os dois construtores alternativos.

In [ ]:
# Escreva sua solução aqui



---
## ☕ Intervalo (15 min)

Até aqui: `@dataclass`, `@classmethod`, `@staticmethod`. Na segunda metade da aula entramos no que diferencia **código orientado a objetos** de **código bem orientado a objetos**: os princípios SOLID e alguns padrões de projeto.

---

## 4. Princípios SOLID (introdução)

**SOLID** é um acrônimo para cinco princípios de design orientado a objetos popularizados por Robert C. Martin. Eles não são *leis* — são *heurísticas* que, quando aplicadas, tendem a produzir código mais flexível, testável e fácil de manter.

Nesta aula cobriremos apenas os três primeiros (SRP, OCP, LSP). Os outros dois (ISP e DIP) ficam para estudos posteriores em disciplinas de engenharia de software.

| Sigla | Nome | Ideia central |
|---|---|---|
| **S** | Single Responsibility | Cada classe tem **uma única razão para mudar**. |
| **O** | Open/Closed | Classes são abertas para extensão, fechadas para modificação. |
| **L** | Liskov Substitution | Subclasses devem poder **substituir** suas superclasses sem quebrar o código. |
| **I** | Interface Segregation | (não cobrimos hoje) |
| **D** | Dependency Inversion | (não cobrimos hoje) |

### 4.1 SRP — Single Responsibility Principle

> *"A class should have one, and only one, reason to change."*

Uma classe deve ter **uma única responsabilidade**. Se você consegue descrever a classe com a palavra "e" (*"essa classe gerencia lanches **e** salva em arquivo **e** envia e-mail..."*), ela provavelmente viola SRP.

**Violação:**

In [ ]:
# ❌ Uma classe que faz tudo — violação gritante do SRP
class PedidoRuim:
    def __init__(self, cliente):
        self.cliente = cliente
        self.itens = []

    def adicionar(self, item):
        self.itens.append(item)

    def total(self):
        return sum(i.preco for i in self.itens)

    # Responsabilidade 2: formatação de relatório
    def gerar_relatorio_html(self):
        html = f"<h1>Pedido de {self.cliente}</h1><ul>"
        for i in self.itens:
            html += f"<li>{i.nome}: R${i.preco:.2f}</li>"
        html += f"</ul><p>Total: R${self.total():.2f}</p>"
        return html

    # Responsabilidade 3: persistência
    def salvar_em_arquivo(self, caminho):
        with open(caminho, "w") as f:
            f.write(self.gerar_relatorio_html())

    # Responsabilidade 4: comunicação
    def enviar_por_email(self, destinatario):
        # ... código hipotético de envio de e-mail
        print(f"📧 Enviando relatório para {destinatario}")


# Essa classe muda se: o modelo de dados mudar, o HTML mudar, o formato de
# arquivo mudar, OU o serviço de e-mail mudar. Quatro razões para mudar =
# quatro responsabilidades.
print("❌ Classe com 4 responsabilidades diferentes.")


**Correção (cada responsabilidade em uma classe):**

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Pedido:
    """Responsabilidade 1: representar o pedido (modelo de dados)."""
    cliente: str
    itens: list = field(default_factory=list)

    def adicionar(self, item):
        self.itens.append(item)

    def total(self):
        return sum(i.preco for i in self.itens)


class RelatorioHTML:
    """Responsabilidade 2: gerar HTML a partir de um pedido."""
    @staticmethod
    def gerar(pedido):
        html = f"<h1>Pedido de {pedido.cliente}</h1><ul>"
        for i in pedido.itens:
            html += f"<li>{i.nome}: R${i.preco:.2f}</li>"
        html += f"</ul><p>Total: R${pedido.total():.2f}</p>"
        return html


class ArmazenamentoArquivo:
    """Responsabilidade 3: persistir conteúdo em disco."""
    @staticmethod
    def salvar(conteudo, caminho):
        print(f"  💾 (simulação) Salvando em {caminho}")
        # with open(caminho, "w") as f: f.write(conteudo)


class EnviadorEmail:
    """Responsabilidade 4: enviar conteúdo por e-mail."""
    @staticmethod
    def enviar(conteudo, destinatario):
        print(f"  📧 (simulação) Enviando para {destinatario}")


# ─── Uso: cada peça faz só o que lhe compete ────────────────────────────
@dataclass
class Lanche:
    nome: str
    preco: float

pedido = Pedido("Ana")
pedido.adicionar(Lanche("X-Tudo", 18.50))
pedido.adicionar(Lanche("Cajuína", 5.00))

html = RelatorioHTML.gerar(pedido)
ArmazenamentoArquivo.salvar(html, "/tmp/pedido.html")
EnviadorEmail.enviar(html, "ana@exemplo.com")

print("\n✔ Cada classe tem uma única razão para mudar.")


**Vantagens:** se amanhã o chefe quiser trocar o formato de HTML para PDF, você só mexe em `RelatorioHTML` (ou cria `RelatorioPDF`). Se quiser trocar o armazenamento por S3, só mexe em `ArmazenamentoArquivo`. Cada mudança fica **isolada**.

### 4.2 OCP — Open/Closed Principle

> *"Software entities should be open for extension, but closed for modification."*

Você deve conseguir **adicionar novos comportamentos** ao sistema **sem alterar** o código existente. O mecanismo tradicional para isso é **polimorfismo com classes abstratas** — exatamente o que vimos na Aula 3.

**Violação:** código com `if` encadeados sobre tipos.

In [ ]:
# ❌ Violação do OCP: precisa editar a função toda vez que aparece novo tipo
class Lanche:
    def __init__(self, nome, preco, tipo):
        self.nome = nome
        self.preco = preco
        self.tipo = tipo                      # "chapa", "panela", "liquidificador"


def preparar_ruim(lanche):
    # Para adicionar "forno", precisa editar esta função.
    if lanche.tipo == "chapa":
        print(f"  🔥 Jogando {lanche.nome} na chapa!")
    elif lanche.tipo == "panela":
        print(f"  🍳 Fervendo {lanche.nome} na panela!")
    elif lanche.tipo == "liquidificador":
        print(f"  🧊 Batendo {lanche.nome} no liquidificador!")
    else:
        print(f"  ??? Não sei preparar {lanche.nome}")


preparar_ruim(Lanche("X-Tudo", 18.50, "chapa"))
preparar_ruim(Lanche("Açaí", 15.00, "liquidificador"))
print("\n❌ Adicionar 'forno' exige mexer em preparar_ruim().")


**Correção:** cada tipo de preparo é uma subclasse. A função cliente chama `item.preparar()` polimorficamente e **nunca precisa mudar**.

In [ ]:
from abc import ABC, abstractmethod


class Lanche(ABC):
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    @abstractmethod
    def preparar(self):
        pass


class LancheChapa(Lanche):
    def preparar(self):
        print(f"  🔥 Jogando {self.nome} na chapa!")


class LanchePanela(Lanche):
    def preparar(self):
        print(f"  🍳 Fervendo {self.nome} na panela!")


class LancheLiquidificador(Lanche):
    def preparar(self):
        print(f"  🧊 Batendo {self.nome} no liquidificador!")


def preparar_bom(lanche):
    """Esta função NUNCA mais precisa ser alterada, não importa quantos
    novos tipos de lanche apareçam."""
    lanche.preparar()


preparar_bom(LancheChapa("X-Tudo", 18.50))
preparar_bom(LancheLiquidificador("Açaí", 15.00))

# Amanhã o chefe compra um forno? Basta CRIAR uma nova classe:
class LancheForno(Lanche):
    def preparar(self):
        print(f"  🔥 Assando {self.nome} no forno a 220°C!")

preparar_bom(LancheForno("Pizza", 25.00))
print("\n✔ Sistema aberto para extensão, fechado para modificação.")


### 4.3 LSP — Liskov Substitution Principle

> *"Objects of a superclass should be replaceable with objects of a subclass without breaking the application."*
> *— Barbara Liskov, 1987*

Se `B` é subclasse de `A`, então todo código que espera um `A` deve funcionar corretamente se receber um `B`. Em outras palavras: a subclasse **não pode violar as expectativas** criadas pela superclasse.

Parece óbvio, mas é fácil violar sem perceber. O exemplo mais famoso é o do **quadrado e retângulo**:

In [ ]:
# ❌ Violação clássica do LSP: Quadrado(Retangulo)
class Retangulo:
    def __init__(self, largura, altura):
        self.largura = largura
        self.altura = altura

    def set_largura(self, v):
        self.largura = v

    def set_altura(self, v):
        self.altura = v

    def area(self):
        return self.largura * self.altura


class Quadrado(Retangulo):
    """Um quadrado 'é um' retângulo... ou não?"""
    def __init__(self, lado):
        super().__init__(lado, lado)

    def set_largura(self, v):
        self.largura = v
        self.altura = v                       # mantém invariante do quadrado

    def set_altura(self, v):
        self.largura = v
        self.altura = v


def redimensionar(retangulo):
    """Função que assume: 'é seguro mudar largura e altura independentemente'."""
    retangulo.set_largura(10)
    retangulo.set_altura(5)
    assert retangulo.area() == 50, f"Esperado 50, obtido {retangulo.area()}"


# Com retângulo, funciona
r = Retangulo(3, 4)
redimensionar(r)
print(f"✔ Retângulo: área = {r.area()}")

# Com quadrado, QUEBRA — mesmo sendo "subclasse"
q = Quadrado(3)
try:
    redimensionar(q)
except AssertionError as e:
    print(f"✘ Quadrado quebrou a suposição: {e}")
    print("✔ Isso é violação do LSP.")


O problema não é que `Quadrado` esteja "errado" — é que a relação "quadrado é um retângulo" é verdadeira em matemática, mas **falsa no contexto do código**, porque retângulos têm a *capacidade* de ter largura e altura independentes, e quadrados não.

**Como evitar violações do LSP?**

1. **Subclasses só devem aceitar o que o pai aceita**, nada menos restritivo.
2. **Subclasses devem devolver o que o pai devolve**, nada mais fraco.
3. **Invariantes do pai devem ser preservados** pela subclasse.
4. Se a "é um" relação não se sustenta do ponto de vista **comportamental**, **use composição** em vez de herança.

No exemplo acima, a solução correta é não fazer `Quadrado` herdar de `Retangulo`. Ambos podem ser `Forma` (classe abstrata), e cada um tem sua própria implementação.

### Exercício 4.1
Identifique qual(is) princípio(s) SOLID a classe abaixo viola e **reescreva** o código corrigindo o(s) problema(s). Justifique suas decisões em comentários.

```python
class GerenciadorLanches:
    def __init__(self):
        self.lanches = []

    def adicionar(self, nome, preco):
        self.lanches.append({"nome": nome, "preco": preco})

    def calcular_total_com_desconto(self, tipo_desconto):
        total = sum(l["preco"] for l in self.lanches)
        if tipo_desconto == "nenhum":
            return total
        elif tipo_desconto == "black_friday":
            return total * 0.5
        elif tipo_desconto == "cupom_10":
            return total * 0.9
        elif tipo_desconto == "estudante":
            return total * 0.85

    def imprimir_no_console(self):
        for l in self.lanches:
            print(f"{l['nome']}: R${l['preco']}")

    def salvar_csv(self, caminho):
        with open(caminho, "w") as f:
            for l in self.lanches:
                f.write(f"{l['nome']},{l['preco']}\n")
```

In [ ]:
# Escreva sua solução refatorada aqui
# Dica: pelo menos SRP e OCP estão sendo violados.



---
## 5. Padrões de projeto (introdução Pythônica)

**Padrões de projeto** (*design patterns*) são soluções reutilizáveis para problemas comuns de design orientado a objetos. O livro clássico é o *Design Patterns: Elements of Reusable Object-Oriented Software* (1994), conhecido como **"GoF"** (*Gang of Four*), que catalogou 23 padrões.

Vamos ver dois que aparecem muito no dia a dia, nas suas versões **Pythônicas** — Python, por ter funções de primeira classe e *duck typing*, frequentemente permite implementações mais simples que as do livro original.

### 5.1 Strategy

**Problema:** você tem um algoritmo que varia em tempo de execução. Em vez de `if/elif` sobre um tipo, você encapsula cada variante em sua própria classe (ou função) e "injeta" a escolhida no objeto que vai usá-la.

**Exemplo no food truck:** diferentes políticas de desconto. O `Pedido` não precisa saber *como* cada desconto funciona — só saber que é aplicado chamando `estrategia.aplicar(total)`.

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field


# ─── Estratégias de desconto ─────────────────────────────────────────────
class EstrategiaDesconto(ABC):
    """Interface para qualquer política de desconto."""
    @abstractmethod
    def aplicar(self, total):
        pass


class SemDesconto(EstrategiaDesconto):
    def aplicar(self, total):
        return total


class DescontoPercentual(EstrategiaDesconto):
    def __init__(self, percentual):
        self.percentual = percentual

    def aplicar(self, total):
        return total * (1 - self.percentual / 100)


class DescontoFixo(EstrategiaDesconto):
    def __init__(self, valor):
        self.valor = valor

    def aplicar(self, total):
        return max(0, total - self.valor)


class DescontoCondicional(EstrategiaDesconto):
    """Aplica um desconto só se o total ultrapassar um limite."""
    def __init__(self, limite, percentual):
        self.limite = limite
        self.percentual = percentual

    def aplicar(self, total):
        if total >= self.limite:
            return total * (1 - self.percentual / 100)
        return total


# ─── Contexto que usa a estratégia ───────────────────────────────────────
@dataclass
class Pedido:
    cliente: str
    itens: list = field(default_factory=list)
    desconto: EstrategiaDesconto = field(default_factory=SemDesconto)

    def adicionar(self, preco):
        self.itens.append(preco)

    def total(self):
        subtotal = sum(self.itens)
        return self.desconto.aplicar(subtotal)


# ─── Uso ─────────────────────────────────────────────────────────────────
p1 = Pedido("João", desconto=SemDesconto())
p1.adicionar(18.50); p1.adicionar(12.00); p1.adicionar(15.00)

p2 = Pedido("Maria", desconto=DescontoPercentual(10))
p2.adicionar(18.50); p2.adicionar(12.00); p2.adicionar(15.00)

p3 = Pedido("Bruno", desconto=DescontoFixo(5.00))
p3.adicionar(18.50); p3.adicionar(12.00); p3.adicionar(15.00)

p4 = Pedido("Ana", desconto=DescontoCondicional(limite=40, percentual=15))
p4.adicionar(18.50); p4.adicionar(12.00); p4.adicionar(15.00)

for p in [p1, p2, p3, p4]:
    print(f"  {p.cliente}: R${p.total():.2f}  (estratégia: {type(p.desconto).__name__})")


**Vantagens:**

- Para adicionar um novo tipo de desconto (ex.: `DescontoAniversariante`), basta **criar uma nova classe**. `Pedido` não muda — é o princípio OCP em ação.
- Cada estratégia é **testável isoladamente**.
- Você pode **trocar a estratégia em tempo de execução** (`pedido.desconto = DescontoPercentual(20)`).

**Versão ainda mais Pythônica:** como Python tem funções de primeira classe, em vez de criar uma classe por estratégia, você pode usar simplesmente **funções** (ou lambdas):

In [ ]:
# Alternativa Pythônica: estratégias como funções
@dataclass
class PedidoSimples:
    cliente: str
    itens: list = field(default_factory=list)
    desconto: object = lambda total: total   # default: identidade

    def total(self):
        subtotal = sum(self.itens)
        return self.desconto(subtotal)


# Estratégias como funções ou lambdas
sem_desconto = lambda t: t
desconto_10 = lambda t: t * 0.9
desconto_fixo_5 = lambda t: max(0, t - 5)

p = PedidoSimples("Teste", itens=[18.50, 12.00, 15.00], desconto=desconto_10)
print(f"Com função-estratégia: R${p.total():.2f}")


Qual versão escolher? Depende:

- **Classes** quando a estratégia tem estado próprio, múltiplos métodos, ou precisa ser subclassificada.
- **Funções** quando a estratégia é simples e sem estado.

### 5.2 Factory Method

**Problema:** você tem múltiplas classes relacionadas e quer centralizar a **criação** delas em um lugar só. O código cliente pede "me dê um lanche do tipo X" sem precisar conhecer as classes concretas.

In [ ]:
from abc import ABC, abstractmethod


class Lanche(ABC):
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    @abstractmethod
    def preparar(self):
        pass

    def __str__(self):
        return f"{self.nome} (R${self.preco:.2f})"


class XTudo(Lanche):
    def __init__(self):
        super().__init__("X-Tudo", 18.50)
    def preparar(self):
        print(f"  🔥 {self.nome}: chapa!")


class Dogao(Lanche):
    def __init__(self):
        super().__init__("Dogão", 12.00)
    def preparar(self):
        print(f"  🍳 {self.nome}: panela!")


class Acai(Lanche):
    def __init__(self):
        super().__init__("Açaí 500ml", 15.00)
    def preparar(self):
        print(f"  🧊 {self.nome}: liquidificador!")


class LancheFactory:
    """Fábrica que centraliza a criação de lanches."""

    _registro = {
        "xtudo": XTudo,
        "dogao": Dogao,
        "acai": Acai,
    }

    @classmethod
    def criar(cls, tipo):
        """Cria um lanche a partir de uma string (ex.: de um menu, formulário...)."""
        if tipo not in cls._registro:
            raise ValueError(f"Tipo de lanche desconhecido: {tipo!r}")
        return cls._registro[tipo]()

    @classmethod
    def registrar(cls, nome, classe):
        """Permite registrar novos tipos em tempo de execução."""
        cls._registro[nome] = classe


# ─── Uso: o cliente não importa as classes concretas ────────────────────
pedidos_brutos = ["xtudo", "dogao", "acai", "xtudo"]
pedido_objetos = [LancheFactory.criar(t) for t in pedidos_brutos]

print("=== Cozinha ===")
for item in pedido_objetos:
    print(f"  Preparando {item}")
    item.preparar()

# Estender sem modificar: registrar um novo tipo dinamicamente
class Tapioca(Lanche):
    def __init__(self):
        super().__init__("Tapioca de queijo", 10.00)
    def preparar(self):
        print(f"  🫓 {self.nome}: chapa baixa!")

LancheFactory.registrar("tapioca", Tapioca)

nova = LancheFactory.criar("tapioca")
nova.preparar()


**Vantagens:**

- O cliente pede por **string** (`"xtudo"`) — perfeito para traduzir dados vindos de formulários, APIs, JSON, CSV, banco de dados.
- Centraliza a lógica de criação em um lugar só.
- Novos tipos podem ser **registrados dinamicamente** sem alterar a fábrica.
- Combina muito bem com `@classmethod`, como vimos.

### Exercício 5.1
Implemente uma `EstrategiaFrete` com pelo menos três estratégias:

- `FreteGratis` (sempre 0).
- `FreteFixo(valor)` (sempre o mesmo valor).
- `FretePorDistancia(taxa_por_km)` que recebe `km` e multiplica.

Adicione a estratégia como atributo do `Pedido` e calcule o total com frete.

In [ ]:
# Escreva sua solução aqui



---
## 6. Estudo de caso integrador: refatorando o Podrão

Você acaba de ser contratado para modernizar o sistema do Podrão. O código atual foi escrito **em modo procedural** pelo sobrinho do dono, e está neste estado:

In [ ]:
# Sistema "legado" do Podrão — 100% procedural, tudo em variáveis soltas

cardapio_nomes = ["X-Tudo", "Dogão", "Açaí 500ml", "Cajuína"]
cardapio_precos = [18.50, 12.00, 15.00, 5.00]
cardapio_tipos = ["chapa", "panela", "liquidificador", "bebida"]

pedidos = []   # cada pedido é uma tupla (cliente, lista_de_indices, pagamento)

def adicionar_pedido(cliente, indices, pagamento):
    pedidos.append((cliente, indices, pagamento))

def total_do_pedido(pedido):
    _, indices, _ = pedido
    return sum(cardapio_precos[i] for i in indices)

def aplicar_desconto(total, tipo_cliente):
    if tipo_cliente == "estudante":
        return total * 0.85
    elif tipo_cliente == "idoso":
        return total * 0.8
    else:
        return total

def preparar_item(indice):
    tipo = cardapio_tipos[indice]
    nome = cardapio_nomes[indice]
    if tipo == "chapa":
        print(f"  🔥 {nome}: chapa!")
    elif tipo == "panela":
        print(f"  🍳 {nome}: panela!")
    elif tipo == "liquidificador":
        print(f"  🧊 {nome}: liquidificador!")
    elif tipo == "bebida":
        print(f"  🥤 {nome}: geladeira!")

def processar_pagamento(total, tipo):
    if tipo == "pix":
        print(f"  📱 Pix: R${total:.2f}")
    elif tipo == "dinheiro":
        print(f"  💵 Dinheiro: R${total:.2f}")
    elif tipo == "fiado":
        print(f"  📝 Fiado: R${total:.2f}")

# Turno simulado
adicionar_pedido("João", [0, 1, 3], "pix")
adicionar_pedido("Maria", [2, 3], "dinheiro")

for p in pedidos:
    cliente, indices, pagamento = p
    print(f"\n--- Pedido de {cliente} ---")
    for i in indices:
        preparar_item(i)
    total = total_do_pedido(p)
    total_com_desconto = aplicar_desconto(total, "estudante" if cliente == "Maria" else "nenhum")
    processar_pagamento(total_com_desconto, pagamento)


**Problemas com esse código:**

1. **Sopa de variáveis**: `cardapio_nomes`, `cardapio_precos`, `cardapio_tipos` são três listas paralelas — um erro silencioso espera para acontecer.
2. **Violação do SRP**: não há separação entre dados, lógica de negócio, cozinha e pagamento.
3. **Violação do OCP**: cada `if/elif` sobre tipo é um convite a editar código existente ao adicionar um novo item.
4. **Impossível de estender**: adicionar "estudante de fim de semana tem desconto maior" exige editar `aplicar_desconto`.
5. **Impossível de testar**: os pedidos moram em uma variável global.
6. **Sem encapsulamento**: tudo é acessível e modificável por qualquer código.

Abaixo está uma **versão refatorada** aplicando o que aprendemos nas quatro aulas. Estude-a com atenção — ela é o "gabarito" do módulo.

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field


# ═══ MODELO DE DOMÍNIO ═══════════════════════════════════════════════════

class Lanche(ABC):
    """Abstração base: todo item do cardápio."""

    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    @abstractmethod
    def preparar(self):
        pass

    def __str__(self):
        return f"{self.nome} (R${self.preco:.2f})"


class LancheChapa(Lanche):
    def preparar(self): print(f"  🔥 {self.nome}: chapa!")

class LanchePanela(Lanche):
    def preparar(self): print(f"  🍳 {self.nome}: panela!")

class LancheLiquidificador(Lanche):
    def preparar(self): print(f"  🧊 {self.nome}: liquidificador!")

class Bebida(Lanche):
    def preparar(self): print(f"  🥤 {self.nome}: geladeira!")


# ═══ ESTRATÉGIAS DE DESCONTO (Strategy + OCP) ════════════════════════════

class EstrategiaDesconto(ABC):
    @abstractmethod
    def aplicar(self, total): pass

class SemDesconto(EstrategiaDesconto):
    def aplicar(self, total): return total

class DescontoEstudante(EstrategiaDesconto):
    def aplicar(self, total): return total * 0.85

class DescontoIdoso(EstrategiaDesconto):
    def aplicar(self, total): return total * 0.80


# ═══ ESTRATÉGIAS DE PAGAMENTO (Strategy) ═════════════════════════════════

class FormaPagamento(ABC):
    @abstractmethod
    def processar(self, valor): pass

class Pix(FormaPagamento):
    def processar(self, valor): print(f"  📱 Pix: R${valor:.2f}")

class Dinheiro(FormaPagamento):
    def processar(self, valor): print(f"  💵 Dinheiro: R${valor:.2f}")

class Fiado(FormaPagamento):
    def processar(self, valor): print(f"  📝 Fiado: R${valor:.2f}")


# ═══ PEDIDO (composição + SRP) ═══════════════════════════════════════════

@dataclass
class Pedido:
    cliente: str
    pagamento: FormaPagamento
    desconto: EstrategiaDesconto = field(default_factory=SemDesconto)
    itens: list = field(default_factory=list)

    def adicionar(self, lanche):
        self.itens.append(lanche)

    def subtotal(self):
        return sum(i.preco for i in self.itens)

    def total(self):
        return self.desconto.aplicar(self.subtotal())

    def __len__(self):
        return len(self.itens)


# ═══ COZINHA (SRP: só prepara) ═══════════════════════════════════════════

class Cozinha:
    @staticmethod
    def processar(pedido):
        print(f"\n--- Cozinha: pedido de {pedido.cliente} ---")
        for item in pedido.itens:
            item.preparar()                   # polimorfismo


# ═══ CAIXA (SRP: só cobra e reporta) ═════════════════════════════════════

class Caixa:
    def __init__(self):
        self._total_dia = 0.0
        self._pedidos_processados = 0

    def cobrar(self, pedido):
        valor = pedido.total()
        print(f"\n--- Caixa: pedido de {pedido.cliente} ---")
        print(f"  Subtotal: R${pedido.subtotal():.2f}")
        if not isinstance(pedido.desconto, SemDesconto):
            print(f"  Desconto: {type(pedido.desconto).__name__}")
        print(f"  Total:    R${valor:.2f}")
        pedido.pagamento.processar(valor)
        self._total_dia += valor
        self._pedidos_processados += 1

    def fechar_dia(self):
        print(f"\n{'='*45}")
        print(f"  📊 FECHAMENTO DO DIA")
        print(f"  Pedidos processados: {self._pedidos_processados}")
        print(f"  Faturamento total:   R${self._total_dia:.2f}")
        print(f"{'='*45}")


# ═══ SIMULAÇÃO DE UM TURNO ════════════════════════════════════════════════

caixa = Caixa()
cozinha = Cozinha()

# Pedido 1 — João, pix, sem desconto
p1 = Pedido("João", pagamento=Pix())
p1.adicionar(LancheChapa("X-Tudo", 18.50))
p1.adicionar(LanchePanela("Dogão", 12.00))
p1.adicionar(Bebida("Cajuína", 5.00))

# Pedido 2 — Maria, estudante, dinheiro
p2 = Pedido("Maria", pagamento=Dinheiro(), desconto=DescontoEstudante())
p2.adicionar(LancheLiquidificador("Açaí 500ml", 15.00))
p2.adicionar(Bebida("Cajuína", 5.00))

# Pedido 3 — Seu Zé, idoso, fiado
p3 = Pedido("Seu Zé", pagamento=Fiado(), desconto=DescontoIdoso())
p3.adicionar(LancheChapa("X-Tudo", 18.50))

for pedido in [p1, p2, p3]:
    cozinha.processar(pedido)
    caixa.cobrar(pedido)

caixa.fechar_dia()


**O que foi aplicado:**

| Aula | Conceito | Onde apareceu |
|---|---|---|
| 1 | Classes e instâncias | Todas as classes do domínio |
| 1 | Atributo de classe | (opcional — não usei aqui mas `Caixa.taxa_padrao` seria candidato) |
| 2 | Encapsulamento | `Caixa._total_dia` |
| 2 | `__str__` | `Lanche.__str__` |
| 2 | `__len__` | `Pedido.__len__` |
| 3 | Herança + polimorfismo | Hierarquia de `Lanche`, `FormaPagamento`, `EstrategiaDesconto` |
| 3 | Composição | `Pedido` **tem** `itens`, `pagamento`, `desconto` |
| 3 | Classes abstratas | `ABC` + `@abstractmethod` em três hierarquias |
| 4 | `@dataclass` | `Pedido` |
| 4 | `@staticmethod` | `Cozinha.processar` |
| 4 | SRP | `Pedido` ≠ `Cozinha` ≠ `Caixa` — três responsabilidades isoladas |
| 4 | OCP | Adicionar novo desconto, pagamento ou tipo de lanche = criar classe nova, **sem tocar no código existente** |
| 4 | LSP | Todas as subclasses podem substituir a superclasse sem surpresas |
| 4 | Strategy | `EstrategiaDesconto`, `FormaPagamento` |

**Para comparar**: conte as linhas do código procedural e do refatorado. O refatorado é **maior**, sim — mas cada pedaço faz **uma coisa só**, pode ser **testado isoladamente**, e o sistema pode crescer sem virar uma bola de neve.

---

## 7. Projeto Final do Módulo

Aplique tudo o que você aprendeu nas quatro aulas em um projeto individual. Entrega no Moodle.

### Enunciado
Você deve modelar um **sistema simplificado de biblioteca universitária** respeitando os princípios vistos no módulo. O sistema deve atender aos seguintes requisitos:

**Domínio:**
- `Item` (abstrato) é qualquer coisa que pode ser emprestada: `Livro`, `Revista`, `DVD`. Cada tipo tem suas próprias regras (ex.: livros emprestam por 14 dias, revistas por 7, DVDs por 3).
- `Usuario` (abstrato) pode ser `Aluno`, `Professor`, `Visitante`. Cada tipo tem limites diferentes de empréstimo simultâneo.
- `Emprestimo` liga um `Usuario` a um ou mais `Item`, com data de retirada e data prevista de devolução.

**Requisitos técnicos:**

1. Use pelo menos uma **`@dataclass`**.
2. Use pelo menos uma **classe abstrata** com `@abstractmethod`.
3. Use pelo menos uma **`@property`** com validação.
4. Implemente `__str__` e `__eq__` nas classes principais.
5. Aplique **polimorfismo** em pelo menos um ponto.
6. Use **composição** para modelar o `Emprestimo`.
7. Use **Strategy** para modelar diferentes **políticas de multa** por atraso (ex.: `MultaFixa`, `MultaProgressiva`).
8. Implemente uma classe `Biblioteca` que tenha métodos `emprestar(usuario, itens)` e `devolver(emprestimo)`, respeitando **SRP** (ela não deve saber imprimir HTML nem enviar e-mails).

**Entrega:**
- Notebook `.ipynb` com o código, comentários explicando as decisões e uma simulação de pelo menos 3 empréstimos diferentes.
- Breve texto (em markdown, no próprio notebook) explicando onde cada princípio SOLID aparece.

**Data:** duas semanas a partir desta aula.

In [ ]:
# Comece o projeto final aqui (ou em um notebook separado)



---
## 8. Resumo do módulo inteiro

Esta foi a última aula do módulo de POO. Você agora tem as ferramentas fundamentais de programação orientada a objetos em Python. Abaixo está um mapa de tudo o que cobrimos:

### Aula 1 — Fundamentos
Tudo em Python é objeto, classes, `__init__`, `self`, atributos de instância vs. classe, métodos.

### Aula 2 — Encapsulamento e dunders
`_protegido`, `__privado`, `@property`, setters com validação, propriedades read-only, `__str__`, `__repr__`, `__eq__`, `__lt__`, `__hash__`, `__len__`.

### Aula 3 — Herança, polimorfismo, composição
`super()`, sobrescrita, *duck typing*, composição vs herança, herança múltipla, MRO, classes abstratas (`ABC`, `@abstractmethod`).

### Aula 4 — Boas práticas
`@dataclass`, `@classmethod`, `@staticmethod`, SOLID (SRP, OCP, LSP), Strategy, Factory Method, refatoração.

### O mantra final

Se eu tivesse que resumir tudo em três frases, seriam estas:

1. **Objetos agrupam dados e comportamento relacionados** — e isso sozinho já resolve metade dos problemas de código procedural.
2. **Prefira composição a herança**; use herança apenas quando a relação "é um" for verdadeira tanto conceitualmente quanto comportamentalmente.
3. **Cada classe deve ter uma única razão para mudar** — e você deve poder estender o sistema sem editar o que já existe.

### Próximos passos (fora deste módulo)

- **Testes automatizados** (`unittest`, `pytest`) — POO fica muito mais divertida quando você tem uma rede de segurança.
- **Tipagem estática** (`typing`, `mypy`) — aproveitar as anotações de tipo para detectar erros antes de rodar.
- **Mais padrões de projeto** — Observer, Decorator, Adapter, Template Method.
- **Os outros dois princípios SOLID** — ISP (*Interface Segregation*) e DIP (*Dependency Inversion*).
- **Arquitetura de software** — como organizar pacotes, camadas, domínios.

Boa sorte no projeto final, e bons objetos!
